# BERTurk NLU eğitimi (intent + NER)

Bu notebook `multi-agent-it-support` reposunun NLU modelini eğitir:
veri → şablon bazlı split → BERTurk fine-tune (MLflow) → ONNX int8 → değerlendirme → `nlu-model.zip`.

**Başlamadan önce:** *Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU*. Sonra *Tümünü çalıştır*.
Toplam süre T4'te ~15–25 dk.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "GPU yok: Çalışma zamanı türünü T4 GPU yapın"

## 1. Repo ve bağımlılıklar

In [ ]:
REPO = "https://github.com/beyzanurulu/multi-agent-it-support.git"
!rm -rf /content/repo && git clone --depth 1 -q $REPO /content/repo
%cd /content/repo
# Colab'da torch, transformers, scikit-learn, tokenizers hazır geliyor.
!pip -q install seqeval mlflow onnx onnxruntime

## 2. Veri

Yerelde `generate.py --llm` ile çeşitlendirilmiş ve Label Studio'da doğrulanmış
**`ml/data/clean.jsonl`** dosyanızı yükleyin. Yüklemezseniz (iptal) repodaki şablon verisi kullanılır;
o zaman test seti doğrulanmamış olur ve rapora öyle yazılır.

In [ ]:
import json
import os

from google.colab import files

try:
    up = files.upload()  # ml/data/clean.jsonl seçin
except Exception:
    up = {}
if up:
    name = next(iter(up))
    os.makedirs("ml/data", exist_ok=True)
    with open("ml/data/clean.jsonl", "wb") as f:
        f.write(up[name])
    print("yüklendi:", name)
else:
    print("dosya yüklenmedi, repodaki ml/data/clean.jsonl kullanılacak")

rows = [json.loads(l) for l in open("ml/data/clean.jsonl", encoding="utf-8")]
print(len(rows), "örnek |", sum(r.get("verified", False) for r in rows), "doğrulanmış |",
      sum(r.get("source") == "llm" for r in rows), "LLM ile yeniden ifade |",
      len({t for r in rows for t in r.get("templates", [])}), "şablon")

## 3. Şablon bazlı split

In [ ]:
verified = sum(r.get("verified", False) for r in rows)
# Test seti varsayılan olarak yalnızca doğrulanmış örneklerden oluşur.
ALLOW_UNVERIFIED_TEST = verified < 100
if ALLOW_UNVERIFIED_TEST:
    print("UYARI: yeterli doğrulanmış örnek yok -> test seti doğrulanmamış veriyle kurulacak")
flag = "--allow-unverified-test" if ALLOW_UNVERIFIED_TEST else ""
!python ml/data/split.py ml/data/clean.jsonl ml/data/splits $flag

## 4. Eğitim (MLflow ile)

In [ ]:
EPOCHS, LR, MAX_LEN, BATCH = 5, 3e-5, 96, 32
!python ml/train.py --train ml/data/splits/train.jsonl --val ml/data/splits/val.jsonl \
    --model dbmdz/bert-base-turkish-cased --epochs $EPOCHS --lr $LR --max-len $MAX_LEN \
    --batch-size $BATCH --out ml/artifacts/berturk

## 5. ONNX export + int8 quantization

In [ ]:
!python ml/export_onnx.py ml/artifacts/berturk models/nlu
!ls -lh models/nlu

## 6. Değerlendirme

Aynı test setinde fine-tuned BERTurk (CPU, ONNX int8) ve kural tabanlı baseline karşılaştırılır.
LLM few-shot baseline için `OPENAI_API_KEY` tanımlayıp `--llm` ekleyebilirsiniz.
Latency Colab CPU'sunda ölçülür; sunucuda yeniden ölçülmeli.

In [ ]:
!python ml/evaluate.py ml/data/splits/test.jsonl --onnx models/nlu --report ml/reports/nlu_eval.md
from IPython.display import Markdown

Markdown(open("ml/reports/nlu_eval.md", encoding="utf-8").read())

## 7. Paketle ve indir

In [ ]:
!cd models/nlu && zip -q -j /content/nlu-model.zip model.onnx tokenizer.json labels.json
!zip -q -r /content/nlu-reports.zip ml/reports/nlu_eval.md ml/data/splits/split_report.json mlruns
!ls -lh /content/*.zip
files.download("/content/nlu-model.zip")
files.download("/content/nlu-reports.zip")

## 8. Sonra

1. GitHub → **Releases → Draft a new release** → tag `nlu-v1` → `nlu-model.zip` dosyasını ekle → **Publish**.
   Demo sayfası ve `nlu-service` imajı modeli buradan otomatik indirir (`ml/fetch_model.py`).
2. `nlu-reports.zip` içeriğini (değerlendirme raporu, split raporu, MLflow kayıtları) model kartını doldurmak için paylaşın.